# ✅ GD Studio — Notebook Solusi

> Buka setelah menyelesaikan `gd_starter.ipynb` (atau benar-benar mentok). Bandingkan **pendekatan**, bukan cuma hasil.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Eksperimen 1 — GD Generik 1D

In [ ]:
def loss1(w):
    return (w - 5) ** 2

def grad1(w):
    return 2 * (w - 5)

def gd_1d(loss_fn, grad_fn, w0, lr=0.1, n_steps=50, tol=1e-8):
    jalur = [w0]
    for _ in range(n_steps):
        w = jalur[-1]
        w_baru = w - lr * grad_fn(w)
        jalur.append(w_baru)
        if abs(w_baru - w) < tol:
            break
    return jalur[-1], jalur

w, jalur = gd_1d(loss1, grad1, w0=0.0, lr=0.1, n_steps=200)
print(f'w_akhir={w:.8f}, jumlah step={len(jalur)-1}, loss_akhir={loss1(w):.2e}')

## Eksperimen 2 — Learning Rate Divergen

In [ ]:
print('lr     | w_akhir        | divergen?')
print('-' * 45)
jalur_divergen = None
for lr in [0.01, 0.1, 0.5, 0.99, 1.01, 1.1]:
    w, jalur = gd_1d(loss1, grad1, w0=0.0, lr=lr, n_steps=800, tol=0.0)
    div = abs(w) > 1e6
    if lr == 1.01:
        jalur_divergen = np.array(jalur)
    print(f'{lr:<6} | {w:>14.4f} | {"YA" if div else "tidak"}')

# Kenapa divergen? Gradien di w: g = 2(w-5). Update: w' = w - lr*2(w-5).
# Jarak ke 5 berlipat |1 - 2*lr| setiap step. Jika |1 - 2*lr| > 1 (lr > 1) -> meledak.

In [ ]:
plt.figure(figsize=(10, 5))
ws = np.linspace(-1.5, 11.5, 300)
plt.plot(ws, loss1(ws), 'k-', alpha=0.3)
plt.plot(jalur_divergen, loss1(jalur_divergen), 'ro--', alpha=0.7)
plt.yscale('log')
plt.xlabel('w'); plt.ylabel('J(w), skala log')
plt.title('GD dengan lr=1.01: loss meledak keluar dari mangkuk')
plt.grid(alpha=0.3); plt.show()

## Eksperimen 3 — Titik Awal Berbeda

In [ ]:
w_dari_kiri, _ = gd_1d(loss1, grad1, w0=-10.0, lr=0.1)
w_dari_kanan, _ = gd_1d(loss1, grad1, w0=10.0, lr=0.1)
print(f'dari -10 -> {w_dari_kiri:.8f}')
print(f'dari +10 -> {w_dari_kanan:.8f}')

# Jawaban: sama-sama sampai ke 5. Loss konveks (bentuk mangkuk tunggal)
# -> hanya ada SATU minimum, dan GD selalu menemukannya dari arah mana pun.
# Di loss non-konveks (eksperimen 5) ini TIDAK berlaku.

## Eksperimen 4 — Loss 2D + Contour

In [ ]:
def loss2(p):
    x, y = p
    return (x - 2) ** 2 + 3 * (y + 1) ** 2

def grad2(p):
    x, y = p
    return np.array([2 * (x - 2), 6 * (y + 1)])

def gd_2d(loss_fn, grad_fn, p0, lr=0.1, n_steps=50, tol=1e-8):
    jalur = [np.asarray(p0, dtype=float)]
    for _ in range(n_steps):
        p = jalur[-1]
        p_baru = p - lr * grad_fn(p)
        jalur.append(p_baru)
        if np.linalg.norm(p_baru - p) < tol:
            break
    return jalur[-1], np.array(jalur)

p_akhir, jalur2 = gd_2d(loss2, grad2, np.array([-3.0, 3.0]), lr=0.1)
print('p_akhir:', p_akhir.round(8))

xs = np.linspace(-4, 6, 200); ys = np.linspace(-4, 4, 200)
XX, YY = np.meshgrid(xs, ys)
ZZ = (XX - 2)**2 + 3*(YY + 1)**2
plt.figure(figsize=(9, 6))
plt.contour(XX, YY, ZZ, levels=25, cmap='viridis', alpha=0.6)
plt.plot(jalur2[:, 0], jalur2[:, 1], 'ro-', ms=3, label='jalur GD')
plt.plot(2, -1, 'g*', ms=15, label='minimum (2, -1)')
plt.colorbar(label='J(x, y)')
plt.legend(); plt.title('GD di loss 2D — perhatikan arah y lebih cepat sampai')
plt.xlabel('x'); plt.ylabel('y'); plt.show()

**Jawaban analisis:** arah `y` lebih agresif karena koefisien 3 membuat gradien di y tiga kali lebih besar — langkah y mengecil cepat, sementara x masih melangkah jauh. Di NN nyata, perbedaan skala antar arah (loss "lembah sempit") persis seperti ini — kenapa *optimizer adaptif* (Adam) dan *feature scaling* penting.

## Eksperimen 5 (Bonus) — Non-Konveks

In [ ]:
def loss3(w):
    return w**4 - 3*w**2 + w

def grad3(w):
    return 4*w**3 - 6*w + 1

hasil = {}
for w0 in (-2.0, -1.5, -0.5, 0.0, 0.5, 1.5, 2.0):
    w, _ = gd_1d(loss3, grad3, w0=w0, lr=0.05, n_steps=200, tol=0.0)
    hasil[w0] = (round(w, 3), round(loss3(w), 3))
for w0, (w, l) in hasil.items():
    print(f'w0={w0:>5} -> w_akhir={w:>8} loss={l}')

ws = np.linspace(-2.2, 2.2, 400)
plt.figure(figsize=(10, 5))
plt.plot(ws, loss3(ws), 'k-', alpha=0.5)
for w0, (w, l) in hasil.items():
    plt.plot(w, l, 'ro')
plt.title('Dua minimum lokal berbeda tergantung titik awal')
plt.grid(alpha=0.3); plt.show()

**Jawaban analisis:** GD mendarat di minimum lokal yang *terdekat dari arah titik awal*. Implikasi ke NN: hasil training tergantung inisialisasi bobot — kenapa praktiknya pakai inisialisasi acak yang dikontrol, beberapa seed, dan kadang pretrained model (Bab 13) sebagai titik awal yang bagus.